In [1]:
# ============================================
# Stage 3 – Dual-Token Fusion (RGB + FFT on Input)
# Full fine-tuning with DeepfakeBench metrics
# ============================================

import os, random, torch, torch.nn as nn, torch.optim as optim
import numpy as np
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import torch.fft as fft
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, f1_score, accuracy_score, roc_curve
from scipy.optimize import brentq
from scipy.interpolate import interp1d
from PIL import Image
from io import BytesIO
import warnings
import torch.nn.functional as F
warnings.filterwarnings("ignore")

In [2]:
!pip uninstall -y facenet-pytorch
!pip install facenet-pytorch==2.5.3 --no-deps

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.9/1.9 MB 34.6 MB/s eta 0:00:00


In [3]:
from facenet_pytorch import MTCNN
from PIL import Image
from pathlib import Path
from tqdm import tqdm
import torch
import shutil

device = 'cuda' if torch.cuda.is_available() else 'cpu'

mtcnn = MTCNN(
    image_size=224,
    margin=40,
    device=device,
    post_process=False
)

SRC_ROOT = Path("/kaggle/input/datasets/hooriyamasood/faceforensics-c23/FaceForensics-c40_frames-Split")
DST_ROOT = Path("/kaggle/working/FF_aligned_faces")

splits = ["train", "val", "test"]
manips = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]

# reset output so old bad folders don't remain
if DST_ROOT.exists():
    shutil.rmtree(DST_ROOT)

def save_aligned(src_img_path, dst_img_path):
    try:
        img = Image.open(src_img_path).convert("RGB")
        face = mtcnn(img)

        dst_img_path.parent.mkdir(parents=True, exist_ok=True)

        if face is None:
            img = img.resize((224, 224))
            img.save(dst_img_path)
        else:
            face = face.permute(1, 2, 0).contiguous()
            if face.max() <= 1.5:
                face = face * 255.0
            face = face.clamp(0, 255).byte().cpu().numpy()
            Image.fromarray(face).save(dst_img_path)
    except Exception as e:
        pass

for split in splits:
    # REAL
    real_src = SRC_ROOT / split / "real"
    real_dst = DST_ROOT / split / "real"

    for video_dir in tqdm(sorted(real_src.iterdir()), desc=f"{split} real"):
        if not video_dir.is_dir():
            continue
        for img_path in video_dir.glob("*"):
            if img_path.is_file():
                dst_path = real_dst / video_dir.name / img_path.name
                save_aligned(img_path, dst_path)

    # FAKE
    fake_src = SRC_ROOT / split / "fake"
    fake_dst = DST_ROOT / split / "fake"

    for manip in manips:
        manip_src = fake_src / manip
        manip_dst = fake_dst / manip

        if not manip_src.exists():
            print(f"Missing manip folder: {manip_src}")
            continue

        for video_dir in tqdm(sorted(manip_src.iterdir()), desc=f"{split} {manip}"):
            if not video_dir.is_dir():
                continue
            for img_path in video_dir.glob("*"):
                if img_path.is_file():
                    dst_path = manip_dst / video_dir.name / img_path.name
                    save_aligned(img_path, dst_path)

print("Alignment finished.")

test NeuralTextures: 100%|██████████| 100/100 [01:15<00:00,  1.33it/s]

Alignment finished.


In [4]:
from pathlib import Path

for split in ["train", "val", "test"]:
    root = Path(f"/kaggle/working/FF_aligned_faces/{split}")
    print(f"\n{split.upper()}")
    print("real:", len(list((root / "real").glob("*"))))
    for m in ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]:
        print(m, len(list((root / "fake" / m).glob("*"))))


TRAIN
real: 800
Deepfakes 800
Face2Face 800
FaceSwap 800
NeuralTextures 800

VAL
real: 100
Deepfakes 100
Face2Face 100
FaceSwap 100
NeuralTextures 100

TEST
real: 100
Deepfakes 100
Face2Face 100
FaceSwap 100
NeuralTextures 100


In [5]:
# ---- DeepfakeBench metric helper ----
def calc_metrics(y_true, y_prob):
    y_pred = (y_prob > 0.5).astype(int)
    auc  = roc_auc_score(y_true, y_prob)
    f1   = f1_score(y_true, y_pred)
    acc  = accuracy_score(y_true, y_pred)
    fpr, tpr, _ = roc_curve(y_true, y_prob)
    eer  = brentq(lambda x: 1. - x - interp1d(fpr, tpr)(x), 0., 1.)
    return auc, f1, eer, acc

In [6]:
import random
from pathlib import Path
from PIL import Image
import torch
from torch.utils.data import Dataset, DataLoader, WeightedRandomSampler
import torchvision.transforms.functional as TF
from torchvision import transforms
from io import BytesIO

# --- your JPEG aug ---
class RandomJPEG:
    def __init__(self, quality_min=30, quality_max=100, p=0.7):
        self.quality_min = quality_min
        self.quality_max = quality_max
        self.p = p

    def __call__(self, img: Image.Image):
        if random.random() > self.p:
            return img
        q = random.randint(self.quality_min, self.quality_max)
        buf = BytesIO()
        img.save(buf, format="JPEG", quality=q)
        buf.seek(0)
        return Image.open(buf).convert("RGB")

class FFPPVideoFramesDataset(Dataset):
    def __init__(self, root, compression="c23", manips=None,
                 train=True, p_flip=0.5,
                 jpeg_aug=None, rgb_color_aug=None, rgb_norm=None):
        self.root = Path(root)
        self.comp = compression
        self.train = train
        self.p_flip = p_flip
        self.jpeg_aug = jpeg_aug
        self.rgb_color_aug = rgb_color_aug
        self.rgb_norm = rgb_norm
        self.class_to_idx = {"real": 0, "fake": 1}

        if manips is None:
            manips = ["Deepfakes", "Face2Face", "FaceSwap", "NeuralTextures"]
        self.manips = manips

        self.samples = []  # list of dicts: {label, frames_dir, manip}

        # real videos
        real_base = self.root / "real"
        for vid_dir in sorted(real_base.glob("*")):
            if vid_dir.is_dir():
                self.samples.append({"label": 0, "frames_dir": vid_dir, "manip": "real"})

        # fake videos

        fake_base = self.root / "fake"
        for m in self.manips:
            m_base = fake_base / m
            for vid_dir in sorted(m_base.glob("*")):
                if vid_dir.is_dir():
                    self.samples.append({"label": 1, "frames_dir": vid_dir, "manip": m})

        # pre-list frames for speed (optional but nice)
        self.frame_cache = {}
        for s in self.samples:
            frames = sorted(list(s["frames_dir"].glob("*.jpg")) + list(s["frames_dir"].glob("*.png")) + list(s["frames_dir"].glob("*.jpeg")))
            self.frame_cache[str(s["frames_dir"])] = frames

        print(f"[FFPPVideoFramesDataset] samples={len(self.samples)} | real={sum(x['label']==0 for x in self.samples)} | fake={sum(x['label']==1 for x in self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s = self.samples[idx]
        frames = self.frame_cache[str(s["frames_dir"])]

        # pick one random frame per video (or deterministic if eval)
        if self.train:
            fpath = random.choice(frames)
        else:
            # stable evaluation: always pick the middle frame
            fpath = frames[len(frames)//2]

        img = Image.open(fpath).convert("RGB")

        # shared geometry aug
        if self.train and random.random() < self.p_flip:
            img = TF.hflip(img)

        # shared jpeg
        img_jpeg = self.jpeg_aug(img) if (self.train and self.jpeg_aug is not None) else img

        # RAW view
        x_raw = TF.to_tensor(img_jpeg)

        # RGB view
        img_rgb = self.rgb_color_aug(img_jpeg) if (self.train and self.rgb_color_aug is not None) else img_jpeg
        x_rgb = TF.to_tensor(img_rgb)
        if self.rgb_norm is not None:
            x_rgb = self.rgb_norm(x_rgb)

        y = s["label"]
        return x_rgb, x_raw, y

def make_balanced_sampler(dataset: FFPPVideoFramesDataset, balance_fake_types=True):
    """
    Returns WeightedRandomSampler that:
    - balances real vs fake
    - optionally balances fake types across manipulations
    """
    # Count groups
    real_idxs = [i for i,s in enumerate(dataset.samples) if s["label"] == 0]
    fake_by_manip = {}
    for i,s in enumerate(dataset.samples):
        if s["label"] == 1:
            fake_by_manip.setdefault(s["manip"], []).append(i)

    weights = [0.0] * len(dataset)

    # Target: 50/50 real/fake
    # Give all real samples equal total weight = 0.5
    w_real = 0.5 / max(len(real_idxs), 1)
    for i in real_idxs:
        weights[i] = w_real

    # For fake: total weight = 0.5
    if not balance_fake_types:
        fake_idxs = [i for i,s in enumerate(dataset.samples) if s["label"] == 1]
        w_fake = 0.5 / max(len(fake_idxs), 1)
        for i in fake_idxs:
            weights[i] = w_fake
    else:
        # Split fake weight equally across manip types
        manips = list(fake_by_manip.keys())
        per_manip_total = 0.5 / max(len(manips), 1)
        for m, idxs in fake_by_manip.items():
            w = per_manip_total / max(len(idxs), 1)
            for i in idxs:
                weights[i] = w

    return WeightedRandomSampler(torch.DoubleTensor(weights), num_samples=len(weights), replacement=True)

In [7]:
class GlobalFilter(nn.Module):
    """
    Learnable complex filter applied in frequency domain.
    Expects fixed H,W feature map size (e.g., 28x28).
    """
    def __init__(self, dim, h, w, fp32fft=True):
        super().__init__()
        self.h = h
        self.w = w
        self.fp32fft = fp32fft

        # rfft2 width is (w//2 + 1)
        self.complex_weight = nn.Parameter(
            torch.randn(h, w // 2 + 1, dim, 2, dtype=torch.float32) * 0.02
        )

    def forward(self, x):
        # x: [B,C,H,W] where H,W match self.h,self.w
        B, C, H, W = x.shape
        assert H == self.h and W == self.w, f"Expected {self.h}x{self.w}, got {H}x{W}"

        x = x.permute(0, 2, 3, 1).contiguous()  # [B,H,W,C]

        if self.fp32fft:
            orig_dtype = x.dtype
            x = x.float()

        x_f = torch.fft.rfft2(x, dim=(1, 2), norm="ortho")  # [B,H,W//2+1,C]
        w = torch.view_as_complex(self.complex_weight)      # [H,W//2+1,C]
        x_f = x_f * w

        x = torch.fft.irfft2(x_f, s=(H, W), dim=(1, 2), norm="ortho")  # [B,H,W,C]

        if self.fp32fft:
            x = x.to(orig_dtype)

        x = x.permute(0, 3, 1, 2).contiguous()  # [B,C,H,W]
        return x

In [8]:
class Stage3Hybrid(nn.Module):
    def __init__(self, embed_dim=512, num_heads=8, num_layers=4, use_fft=True, grid=10):
        super().__init__()
        self.use_fft = use_fft
        self.grid = grid
        self.freq_filter = GlobalFilter(dim=512, h=28, w=28, fp32fft=True)

        # --backbone--
        resnet = models.resnet50(weights=models.ResNet50_Weights.IMAGENET1K_V1)

           # split resnet into stages
        self.stem = nn.Sequential(resnet.conv1, resnet.bn1, resnet.relu, resnet.maxpool)
        self.layer1 = resnet.layer1
        self.layer2 = resnet.layer2   # -> [B,512,28,28] for 224 input
        self.layer3 = resnet.layer3
        self.layer4 = resnet.layer4   # -> [B,2048,7,7]
        
            # RGB tokens from layer4 (same spirit as before)
        self.proj_rgb = nn.Conv2d(2048, embed_dim, 1)
        
           # FREQ tokens will come from layer2 (512 channels)
        self.proj_fft = nn.Conv2d(512, embed_dim, 1)

        
        # ---- Cross-Modality Fusion (RGB attends to FFT) ----
        self.cross_attn = nn.MultiheadAttention(embed_dim, num_heads, batch_first=True)

        # ---- Transformer encoder ----
        enc_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            dim_feedforward=embed_dim * 4,
            batch_first=True,
            norm_first=True
        )
        self.transformer = nn.TransformerEncoder(enc_layer, num_layers=num_layers)

        # ---- tokens ----
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.type_embed = nn.Parameter(torch.randn(1, 2, embed_dim))  # 0=RGB, 1=FFT

        # positions: 1 CLS + 2*(grid*grid)
        num_tokens = 1 + 2 * (grid * grid)
        self.pos_embed = nn.Parameter(torch.randn(1, num_tokens, embed_dim))

        # classifier
        self.cls_head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Linear(embed_dim, 2)
        )
   
   
    def forward(self, x_rgb, x_raw=None):
        B = x_rgb.size(0)
    
        # ---- run ResNet stages ----
        x = self.stem(x_rgb)
        x = self.layer1(x)
        feat_l2 = self.layer2(x)        # [B,512,28,28]  <-- use this for freq branch
    
        x = self.layer3(feat_l2)
        feat_l4 = self.layer4(x)        # [B,2048,7,7]   <-- use this for rgb branch
    
        # ---- RGB tokens ----
        rgb_map = self.proj_rgb(feat_l4)                       # [B,E,7,7]
        rgb_map = F.adaptive_avg_pool2d(rgb_map, (self.grid, self.grid))
        rgb_tok = rgb_map.flatten(2).transpose(1, 2)           # [B,G^2,E]
    
        # ---- FREQ tokens (M2TR-style learned filter on feature map) ----
        if self.use_fft:
            freq_feat = self.freq_filter(feat_l2)              # [B,512,28,28]
            fft_map = self.proj_fft(freq_feat)                 # [B,E,28,28]
            fft_map = F.adaptive_avg_pool2d(fft_map, (self.grid, self.grid))
            fft_tok = fft_map.flatten(2).transpose(1, 2)       # [B,G^2,E]
        else:
            fft_tok = torch.zeros_like(rgb_tok)
    
        # ---- type embeds ----
        rgb_tok = rgb_tok + self.type_embed[:, 0:1, :]
        fft_tok = fft_tok + self.type_embed[:, 1:2, :]
    
        # ---- Cross-attn fusion ----
        fused_rgb, _ = self.cross_attn(query=rgb_tok, key=fft_tok, value=fft_tok)
        rgb_tok = rgb_tok + fused_rgb
    
        # ---- tokens + transformer ----
        cls = self.cls_token.repeat(B, 1, 1)
        tokens = torch.cat([cls, rgb_tok, fft_tok], dim=1)
        tokens = tokens + self.pos_embed[:, :tokens.size(1), :]
    
        out = self.transformer(tokens)
        cls_out = out[:, 0]
        return self.cls_head(cls_out)


In [9]:
def evaluate(model, loader, class_to_idx, tag="EVAL"):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model.eval()

    fake_idx = class_to_idx["fake"]  # positive class
    y_true, y_prob = [], []

    with torch.no_grad():
        for x_rgb, x_raw, lbls in tqdm(loader, desc=tag, ncols=100):
            x_rgb = x_rgb.to(device, non_blocking=True)
            logits = model(x_rgb, x_raw=None)
            probs_fake = torch.softmax(logits, dim=1)[:, fake_idx].cpu().numpy()

            lbls_np = lbls.numpy()
            y_true.extend((lbls_np == fake_idx).astype(np.int32))
            y_prob.extend(probs_fake)

    auc, f1, eer, acc = calc_metrics(np.array(y_true), np.array(y_prob))
    print(f"{tag} → AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}")
    return auc, f1, eer, acc

In [10]:
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR


def finetune(pretrained_ckpt_path,
                  train_root, val_root, test_root,
                  epochs=15, freeze_epochs=1, batch_size=16):

    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = Stage3Hybrid().to(device)

    state = torch.load(pretrained_ckpt_path, map_location="cpu")
    model.load_state_dict(state, strict=True)
    print("Loaded pretrained:", pretrained_ckpt_path)

   # ---- augs ----
    jpeg_aug = RandomJPEG(p=0.3, quality_min=60, quality_max=95)
    rgb_color_aug = transforms.ColorJitter(0.08, 0.08, 0.04, 0.02)
    rgb_norm = transforms.Normalize([0.485,0.456,0.406],[0.229,0.224,0.225])
    
    
    trainset = FFPPVideoFramesDataset(train_root, compression="c23", train=True,
                                  jpeg_aug=jpeg_aug, rgb_color_aug=rgb_color_aug, rgb_norm=rgb_norm)
    
    valset   = FFPPVideoFramesDataset(val_root, compression="c23", train=False,
                                      jpeg_aug=None, rgb_color_aug=None, rgb_norm=rgb_norm)
    
    testset  = FFPPVideoFramesDataset(test_root, compression="c23", train=False,
                                      jpeg_aug=None, rgb_color_aug=None, rgb_norm=rgb_norm)
    
    # Balanced sampler for training

    sampler = make_balanced_sampler(trainset, balance_fake_types=True)
    trainloader = DataLoader(trainset, batch_size=batch_size, sampler=sampler, num_workers=2)
    valloader   = DataLoader(valset,   batch_size=16, shuffle=False, num_workers=2, pin_memory=True)
    testloader  = DataLoader(testset,  batch_size=16, shuffle=False, num_workers=2, pin_memory=True)

    class_to_idx = trainset.class_to_idx
    print("class_to_idx:", class_to_idx)
    fake_idx = class_to_idx["fake"]

    # backbone/head params
    backbone_params = (
        list(model.stem.parameters()) +
        list(model.layer1.parameters()) +
        list(model.layer2.parameters()) +
        list(model.layer3.parameters()) +
        list(model.layer4.parameters())
    )
    
    head_params = [
        p for n, p in model.named_parameters()
        if not (
            n.startswith("stem.") or
            n.startswith("layer1.") or
            n.startswith("layer2.") or
            n.startswith("layer3.") or
            n.startswith("layer4.")
        )
    ]
    
    opt = optim.AdamW([
        {"params": backbone_params, "lr": 1e-5},
        {"params": head_params,     "lr": 7e-5},
    ], weight_decay=5e-4)
    
    scheduler = CosineAnnealingLR(opt, T_max=epochs)
    criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
    
    use_cuda = torch.cuda.is_available()
    scaler = torch.amp.GradScaler(enabled=use_cuda)

    best_auc = -1.0

    for epoch in range(epochs):
        # freeze warmup
        if epoch < freeze_epochs:
            for p in backbone_params:
                p.requires_grad = False
        else:
            for p in backbone_params:
                p.requires_grad = True

        model.train()
        running_loss = 0.0

        for x_rgb, x_raw, lbls in tqdm(trainloader, desc=f"FT {epoch+1}/{epochs}", ncols=100):
            x_rgb = x_rgb.to(device, non_blocking=True)
            lbls  = lbls.to(device, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            with torch.amp.autocast(device_type="cuda", enabled=use_cuda):
                logits = model(x_rgb, x_raw=None)
                loss = criterion(logits, lbls)

            scaler.scale(loss).backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(opt)
            scaler.update()

            running_loss += float(loss.item())

        # validate (fake positive)
        model.eval()
        y_true, y_prob = [], []
        with torch.no_grad():
            for x_rgb, x_raw, lbls in valloader:
                x_rgb = x_rgb.to(device, non_blocking=True)
                logits = model(x_rgb, x_raw=None)
                probs_fake = torch.softmax(logits, dim=1)[:, fake_idx].cpu().numpy()
                lbls_np = lbls.numpy()
                y_true.extend((lbls_np == fake_idx).astype(np.int32))
                y_prob.extend(probs_fake)

        auc, f1, eer, acc = calc_metrics(np.array(y_true), np.array(y_prob))
        print(f"Epoch {epoch+1}: loss={running_loss/len(trainloader):.4f} | "
              f"AUROC={auc:.3f} | F1={f1:.3f} | EER={eer:.3f} | ACC={acc:.3f}")

        # step scheduler once per epoch
        scheduler.step()
        print("LRs:", [pg["lr"] for pg in opt.param_groups])

        # save best
        if auc > best_auc:
            best_auc = auc
            torch.save(model.state_dict(), "/kaggle/working/best_auc_stage3_finetune.pth")
            print(f"★ New best AUC {best_auc:.3f}")

    # final test with best
    model.load_state_dict(torch.load("/kaggle/working/best_auc_stage3_finetune.pth", map_location=device))
    evaluate(model, testloader, class_to_idx, tag="FF++ TEST")

    return model, class_to_idx

In [11]:
from pathlib import Path
pretrained_ckpt = "/kaggle/input/models/hooriyamasood/140k-m2tr-face-alignment/pytorch/default/1/best_auc_stage3.pth" 
train_root = "/kaggle/working/FF_aligned_faces/train"
val_root   = "/kaggle/working/FF_aligned_faces/val"
test_root  = "/kaggle/working/FF_aligned_faces/test"


print("real videos:", len(list(Path(train_root + "/real").glob("*"))))
print("fake deepfakes:", len(list(Path(train_root + "/fake/Deepfakes").glob("*"))))

model, class_to_idx = finetune(pretrained_ckpt, train_root, val_root, test_root)

real videos: 800
fake deepfakes: 800
Downloading: "https://download.pytorch.org/models/resnet50-0676ba61.pth" to /root/.cache/torch/hub/checkpoints/resnet50-0676ba61.pth


100%|██████████| 97.8M/97.8M [00:00<00:00, 157MB/s]


Loaded pretrained: /kaggle/input/models/hooriyamasood/140k-m2tr-face-alignment/pytorch/default/1/best_auc_stage3.pth
[FFPPVideoFramesDataset] samples=4000 | real=800 | fake=3200
[FFPPVideoFramesDataset] samples=500 | real=100 | fake=400
[FFPPVideoFramesDataset] samples=500 | real=100 | fake=400
class_to_idx: {'real': 0, 'fake': 1}


FT 1/15: 100%|████████████████████████████████████████████████████| 250/250 [00:27<00:00,  9.21it/s]


Epoch 1: loss=0.8674 | AUROC=0.567 | F1=0.759 | EER=0.420 | ACC=0.644
LRs: [9.890738003669029e-06, 6.923516602568319e-05]
★ New best AUC 0.567


...

Epoch 14: loss=0.3511 | AUROC=0.906 | F1=0.912 | EER=0.160 | ACC=0.862
LRs: [1.092619963309716e-07, 7.648339743168008e-07]


FT 15/15: 100%|███████████████████████████████████████████████████| 250/250 [00:41<00:00,  6.03it/s]


Epoch 15: loss=0.3595 | AUROC=0.912 | F1=0.903 | EER=0.168 | ACC=0.850
LRs: [0.0, 0.0]
★ New best AUC 0.912


FF++ TEST: 100%|████████████████████████████████████████████████████| 32/32 [00:01<00:00, 19.01it/s]


FF++ TEST → AUROC=0.900 | F1=0.893 | EER=0.190 | ACC=0.836
